# 163. Multi-Head Latent Attention：怎样实现 MLA，并证明 latent KV Cache 的解码等价性？

> **面试问题：DeepSeek 风格 MLA 到底压缩了什么？低秩 KV、解耦 RoPE、矩阵吸收和增量缓存怎样从零实现？**

## 先给结论

MLA 不是简单把 KV 做完再量化，而是让每个 token 只缓存共享的低维 KV latent 与位置分量，需要时再上投影出各头的 key/value。真正答好要同时说清：低秩瓶颈的表达代价、RoPE 为什么要解耦、训练全序列与逐 token decode 的等价 oracle、缓存字节，以及实现是否真的做了矩阵吸收。

## 推荐的回答主线

1. 先以每 token 缓存状态定义问题，区分普通 MHA/GQA 的 per-head K/V 与 MLA 的共享 latent。
2. 写出 down/up projection、content key 与 positional key 拼接、causal softmax 和输出投影的张量形状。
3. 用同一组权重逐 token 重建 K/V，验证 prefill 与 decode 输出逐位置一致。
4. 再讨论矩阵吸收、SVD 转换、实际 kernel/量化、模型配置与 cache 版本门禁。

## 本 Notebook 的实现边界

这里用小型 PyTorch 模块显式重建 KV，重点验证数学和缓存合同；没有实现 FlashMLA、FP8 kernel、张量并行或特定 checkpoint 的完整细节。吞吐收益必须在目标 batch、序列长度和硬件上测量。

## 一手资料

- [DeepSeek-V2 / MLA](https://arxiv.org/abs/2405.04434)
- [MHA2MLA](https://arxiv.org/abs/2502.14837)
- [Hardware-Efficient MLA](https://arxiv.org/abs/2506.02523)


In [ ]:
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
from dataclasses import asdict, dataclass  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

# 固定随机性并声明所有形状，面试时先把符号和轴讲清楚。
torch.manual_seed(163)  # 执行当前语句以推进本节示例。
B, L, D_MODEL = 2, 6, 12  # 计算并保存当前步骤的中间状态。
N_HEADS, D_NOPE, D_ROPE, D_VALUE, KV_RANK, Q_RANK = 3, 3, 2, 4, 5, 6  # 计算并保存当前步骤的中间状态。
x = torch.randn(B, L, D_MODEL)  # 计算并保存当前步骤的中间状态。

assert D_ROPE % 2 == 0  # 用受控断言验证关键不变量。
assert x.shape == (B, L, D_MODEL)  # 用受控断言验证关键不变量。
assert KV_RANK < N_HEADS * (D_NOPE + D_VALUE)  # 用受控断言验证关键不变量。


## 1. 先单测 RoPE：位置只旋转成对维度

MLA 常把与内容相关的低秩 key 和承载位置信息的 RoPE key 分开。这样缓存中无需为每个 query head 保存完整旋转 key。下面只实现最小 RoPE；范数不变和位置 0 恒等是很好的局部 oracle。


In [ ]:
def apply_rope(z, positions, base=10_000.0):  # 定义本节可复用的核心函数。
    """z: [B,H,L,D_rope]；positions: [L]。"""  # 执行当前语句以推进本节示例。
    half = z.shape[-1] // 2  # 计算并保存当前步骤的中间状态。
    inv_freq = base ** (-torch.arange(half, dtype=z.dtype, device=z.device) / half)  # 计算并保存当前步骤的中间状态。
    angle = positions.to(z.dtype)[:, None] * inv_freq[None, :]  # 计算并保存当前步骤的中间状态。
    cos, sin = angle.cos()[None, None], angle.sin()[None, None]  # 计算并保存当前步骤的中间状态。
    left, right = z[..., :half], z[..., half:]  # 计算并保存当前步骤的中间状态。
    return torch.cat([left * cos - right * sin, left * sin + right * cos], dim=-1)  # 返回当前分支计算出的结果。

# 旋转应保持每个向量的二范数，且零位置不改变输入。
probe = torch.randn(B, N_HEADS, L, D_ROPE)  # 计算并保存当前步骤的中间状态。
rotated = apply_rope(probe, torch.arange(L))  # 计算并保存当前步骤的中间状态。
assert rotated.shape == probe.shape  # 用受控断言验证关键不变量。
assert torch.allclose(rotated.norm(dim=-1), probe.norm(dim=-1), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(rotated[:, :, 0], probe[:, :, 0], atol=1e-6)  # 用受控断言验证关键不变量。


## 2. 手写 TinyMLA.forward：content 与 position 分路

查询也可先下投影到低秩空间，再分别上投影为 non-RoPE 与 RoPE 部分。KV 侧每个 token 生成一个共享 latent；content key/value 从 latent 重建，位置 key 则由输入单独产生并广播到各头。教学实现故意保留显式重建，方便看清形状。


In [ ]:
class TinyMLA(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.q_down = nn.Linear(D_MODEL, Q_RANK, bias=False)  # 计算并保存当前步骤的中间状态。
        self.q_nope = nn.Linear(Q_RANK, N_HEADS * D_NOPE, bias=False)  # 计算并保存当前步骤的中间状态。
        self.q_rope = nn.Linear(Q_RANK, N_HEADS * D_ROPE, bias=False)  # 计算并保存当前步骤的中间状态。
        self.kv_down = nn.Linear(D_MODEL, KV_RANK, bias=False)  # 计算并保存当前步骤的中间状态。
        self.k_up = nn.Linear(KV_RANK, N_HEADS * D_NOPE, bias=False)  # 计算并保存当前步骤的中间状态。
        self.v_up = nn.Linear(KV_RANK, N_HEADS * D_VALUE, bias=False)  # 计算并保存当前步骤的中间状态。
        self.k_rope = nn.Linear(D_MODEL, D_ROPE, bias=False)  # 计算并保存当前步骤的中间状态。
        self.out = nn.Linear(N_HEADS * D_VALUE, D_MODEL, bias=False)  # 计算并保存当前步骤的中间状态。

    def project(self, tokens, positions):  # 定义本节可复用的核心函数。
        b, length, _ = tokens.shape  # 计算并保存当前步骤的中间状态。
        q_latent = self.q_down(tokens)  # 计算并保存当前步骤的中间状态。
        qn = self.q_nope(q_latent).view(b, length, N_HEADS, D_NOPE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        qr = self.q_rope(q_latent).view(b, length, N_HEADS, D_ROPE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        latent = self.kv_down(tokens)  # 计算并保存当前步骤的中间状态。
        kn = self.k_up(latent).view(b, length, N_HEADS, D_NOPE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        value = self.v_up(latent).view(b, length, N_HEADS, D_VALUE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        kr = self.k_rope(tokens)[:, None].expand(-1, N_HEADS, -1, -1)  # 计算并保存当前步骤的中间状态。
        return torch.cat([qn, apply_rope(qr, positions)], -1), latent, torch.cat([kn, apply_rope(kr, positions)], -1), value  # 返回当前分支计算出的结果。

    def forward(self, tokens, valid=None):  # 定义本节可复用的核心函数。
        length = tokens.shape[1]  # 计算并保存当前步骤的中间状态。
        q, _, k, value = self.project(tokens, torch.arange(length, device=tokens.device))  # 计算并保存当前步骤的中间状态。
        scores = q @ k.transpose(-1, -2) / math.sqrt(D_NOPE + D_ROPE)  # 计算并保存当前步骤的中间状态。
        allowed = torch.ones(length, length, dtype=torch.bool, device=tokens.device).tril()  # 计算并保存当前步骤的中间状态。
        if valid is not None:  # 按当前条件选择后续控制路径。
            allowed = allowed[None, None] & valid[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        scores = scores.masked_fill(~allowed, -torch.inf)  # 计算并保存当前步骤的中间状态。
        context = torch.softmax(scores, -1) @ value  # 计算并保存当前步骤的中间状态。
        return self.out(context.transpose(1, 2).reshape(tokens.shape[0], length, -1))  # 返回当前分支计算出的结果。

# 模块必须保持 batch/sequence 轴，并产生有限值。
mla = TinyMLA().eval()  # 计算并保存当前步骤的中间状态。
full = mla(x)  # 计算并保存当前步骤的中间状态。
assert full.shape == x.shape  # 用受控断言验证关键不变量。
assert torch.isfinite(full).all()  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in mla.parameters()) > 0  # 用受控断言验证关键不变量。


## 3. 增量 decode：缓存 latent，而不是缓存重建后的每头 KV

逐 token 时把当前 token 的 KV latent 和位置 key 追加到缓存，再用同一上投影重建历史 K/V。只要位置、mask、权重和数值路径一致，第 t 步输出应等于全序列 forward 的第 t 个位置。这是重构 kernel 前最有价值的回归测试。


In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def decode_one_by_one(model, tokens):  # 定义本节可复用的核心函数。
    latent_cache, rope_cache, outputs = [], [], []  # 计算并保存当前步骤的中间状态。
    for t in range(tokens.shape[1]):  # 遍历输入元素以累积或检查结果。
        current = tokens[:, t:t + 1]  # 计算并保存当前步骤的中间状态。
        q, latent, _, _ = model.project(current, torch.tensor([t]))  # 计算并保存当前步骤的中间状态。
        latent_cache.append(latent)  # 执行当前语句以推进本节示例。
        raw_rope = model.k_rope(current)  # 计算并保存当前步骤的中间状态。
        rope_cache.append(apply_rope(raw_rope[:, None], torch.tensor([t])))  # 执行当前语句以推进本节示例。
        all_latent = torch.cat(latent_cache, dim=1)  # 计算并保存当前步骤的中间状态。
        kn = model.k_up(all_latent).view(B, t + 1, N_HEADS, D_NOPE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        value = model.v_up(all_latent).view(B, t + 1, N_HEADS, D_VALUE).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        k = torch.cat([kn, torch.cat(rope_cache, dim=2).expand(-1, N_HEADS, -1, -1)], dim=-1)  # 计算并保存当前步骤的中间状态。
        weights = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(D_NOPE + D_ROPE), dim=-1)  # 计算并保存当前步骤的中间状态。
        ctx = weights @ value  # 计算并保存当前步骤的中间状态。
        outputs.append(model.out(ctx.transpose(1, 2).reshape(B, 1, -1)))  # 执行当前语句以推进本节示例。
    return torch.cat(outputs, dim=1), torch.cat(latent_cache, dim=1), torch.cat(rope_cache, dim=2)  # 返回当前分支计算出的结果。

# 全序列与逐 token 路径应逐位置等价，缓存只含共享 latent 和位置 key。
decoded, latent_cache, rope_cache = decode_one_by_one(mla, x)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(decoded, full, atol=1e-6)  # 用受控断言验证关键不变量。
assert latent_cache.shape == (B, L, KV_RANK)  # 用受控断言验证关键不变量。
assert rope_cache.shape == (B, 1, L, D_ROPE)  # 用受控断言验证关键不变量。


## 4. 缓存字节：必须把层数、token 数、dtype 都写进公式

普通 MHA 每层每 token 保存 H 份 K 和 H 份 V；教学版 MLA 保存一个 KV latent 与一个共享位置 key。比值只是架构级估计，真实系统还含块表、对齐、scale、残差窗口和分片副本。


In [ ]:
def cache_bytes(tokens, layers, batch, elements_per_token, bytes_per_element=2):  # 定义本节可复用的核心函数。
    return tokens * layers * batch * elements_per_token * bytes_per_element  # 返回当前分支计算出的结果。

tokens, layers, batch = 32_768, 32, 4  # 计算并保存当前步骤的中间状态。
mha_elements = N_HEADS * ((D_NOPE + D_ROPE) + D_VALUE)  # 计算并保存当前步骤的中间状态。
mla_elements = KV_RANK + D_ROPE  # 计算并保存当前步骤的中间状态。
mha_bytes = cache_bytes(tokens, layers, batch, mha_elements)  # 计算并保存当前步骤的中间状态。
mla_bytes = cache_bytes(tokens, layers, batch, mla_elements)  # 计算并保存当前步骤的中间状态。

# 公式应随 token 线性增长，且本配置的 latent cache 小于完整 MHA cache。
assert cache_bytes(tokens * 2, layers, batch, mla_elements) == 2 * mla_bytes  # 用受控断言验证关键不变量。
assert mla_bytes < mha_bytes  # 用受控断言验证关键不变量。
assert math.isclose(mha_bytes / mla_bytes, mha_elements / mla_elements)  # 用受控断言验证关键不变量。


## 5. 矩阵吸收：证明 content score 可在 latent 空间计算

若暂不考虑 RoPE 分量，`(c_q W_q)(c_k W_k)^T` 可重排为 `c_q (W_q W_k^T) c_k^T`。这叫代数吸收；它减少显式上投影的中间量，但能否更快还取决于维度、融合 kernel 和硬件。位置部分仍需单独处理。


In [ ]:
# 从 Linear 的 [out,in] 权重转成右乘形式，直接验证结合律。
cq = torch.randn(B, 2, Q_RANK)  # 计算并保存当前步骤的中间状态。
ck = torch.randn(B, 4, KV_RANK)  # 计算并保存当前步骤的中间状态。
wq = mla.q_nope.weight.view(N_HEADS, D_NOPE, Q_RANK)  # 计算并保存当前步骤的中间状态。
wk = mla.k_up.weight.view(N_HEADS, D_NOPE, KV_RANK)  # 计算并保存当前步骤的中间状态。
explicit = torch.einsum("btr,hdr->bhtd", cq, wq) @ torch.einsum("bsr,hdr->bhsd", ck, wk).transpose(-1, -2)  # 计算并保存当前步骤的中间状态。
absorbed_matrix = torch.einsum("hdr,hdk->hrk", wq, wk)  # 计算并保存当前步骤的中间状态。
absorbed = torch.einsum("btr,hrk,bsk->bhts", cq, absorbed_matrix, ck)  # 计算并保存当前步骤的中间状态。

# 两条路径只应有浮点舍入差，吸收矩阵按 head 独立。
assert explicit.shape == (B, N_HEADS, 2, 4)  # 用受控断言验证关键不变量。
assert absorbed_matrix.shape == (N_HEADS, Q_RANK, KV_RANK)  # 用受控断言验证关键不变量。
assert torch.allclose(explicit, absorbed, atol=1e-5)  # 用受控断言验证关键不变量。


## 6. 从 MHA 转换：SVD 只能给近似初始化，不能保证无损

把已有大投影压到 rank-r，可用截断 SVD 初始化 down/up projection。最佳低秩近似误差随 rank 增大不应变差，但模型质量仍要校准或继续训练；不能把矩阵 Frobenius 误差当最终生成质量。


In [ ]:
def truncated_svd(matrix, rank):  # 定义本节可复用的核心函数。
    u, s, vh = torch.linalg.svd(matrix, full_matrices=False)  # 计算并保存当前步骤的中间状态。
    down = u[:, :rank] * s[:rank]  # 计算并保存当前步骤的中间状态。
    up = vh[:rank]  # 计算并保存当前步骤的中间状态。
    return down, up, down @ up  # 返回当前分支计算出的结果。

# 用同一完整投影比较不同 rank 的最优近似误差。
full_projection = torch.randn(D_MODEL, N_HEADS * (D_NOPE + D_VALUE))  # 计算并保存当前步骤的中间状态。
d2, u2, approx2 = truncated_svd(full_projection, 2)  # 计算并保存当前步骤的中间状态。
d5, u5, approx5 = truncated_svd(full_projection, 5)  # 计算并保存当前步骤的中间状态。
err2 = (full_projection - approx2).norm()  # 计算并保存当前步骤的中间状态。
err5 = (full_projection - approx5).norm()  # 计算并保存当前步骤的中间状态。
assert d5.shape == (D_MODEL, 5)  # 用受控断言验证关键不变量。
assert u5.shape == (5, full_projection.shape[1])  # 用受控断言验证关键不变量。
assert err5 <= err2 + 1e-6  # 用受控断言验证关键不变量。


## 7. Padding 与 causal mask：无效 key 不得获得概率

批量 prefill 会同时出现 causal 与 padding 两类约束。全为 `-inf` 的 query 行会产生 NaN，因此生产实现还需 query mask 或空行策略；这里保留有效 query，只验证 padding key 与未来 key 均不泄漏。


In [ ]:
def attention_probabilities(model, tokens, valid):  # 定义本节可复用的核心函数。
    length = tokens.shape[1]  # 计算并保存当前步骤的中间状态。
    q, _, k, _ = model.project(tokens, torch.arange(length))  # 计算并保存当前步骤的中间状态。
    score = q @ k.transpose(-1, -2) / math.sqrt(D_NOPE + D_ROPE)  # 计算并保存当前步骤的中间状态。
    allowed = torch.ones(length, length, dtype=torch.bool).tril()[None, None]  # 计算并保存当前步骤的中间状态。
    allowed = allowed & valid[:, None, None, :]  # 计算并保存当前步骤的中间状态。
    return torch.softmax(score.masked_fill(~allowed, -torch.inf), -1)  # 返回当前分支计算出的结果。

# 第一条样本后两个位置是 padding；有效查询不能看到它们或未来 token。
valid = torch.tensor([[1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
prob = attention_probabilities(mla, x, valid)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(prob[0, :, :4, 4:], torch.zeros_like(prob[0, :, :4, 4:]))  # 用受控断言验证关键不变量。
assert prob[0, :, 3, :4].sum(-1).allclose(torch.ones(N_HEADS))  # 用受控断言验证关键不变量。
assert prob[1, :, 0, 1:].max().item() == 0.0  # 用受控断言验证关键不变量。


## 8. 制品门禁：模型结构和 cache layout 必须共同版本化

若服务端拿错 rank、head 数、RoPE 维度或权重摘要，缓存虽能分配却会产生静默错误。生产中应让 checkpoint、kernel 和 cache manager 对同一配置指纹达成一致，再做真实 TTFT/TPOT、显存峰值和质量回归。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class MLAConfig:  # 定义承载本节状态与行为的数据结构。
    model_dim: int  # 执行当前语句以推进本节示例。
    heads: int  # 执行当前语句以推进本节示例。
    kv_rank: int  # 执行当前语句以推进本节示例。
    rope_dim: int  # 执行当前语句以推进本节示例。
    value_dim: int  # 执行当前语句以推进本节示例。
    cache_dtype: str = "float16"  # 计算并保存当前步骤的中间状态。

def fingerprint(config):  # 定义本节可复用的核心函数。
    payload = json.dumps(asdict(config), sort_keys=True).encode()  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload).hexdigest()[:16]  # 返回当前分支计算出的结果。

# cache header 必须与加载模型的结构指纹完全一致。
config = MLAConfig(D_MODEL, N_HEADS, KV_RANK, D_ROPE, D_VALUE)  # 计算并保存当前步骤的中间状态。
cache_header = {"layout": "latent_plus_rope_v1", "config_hash": fingerprint(config)}  # 计算并保存当前步骤的中间状态。
assert len(cache_header["config_hash"]) == 16  # 用受控断言验证关键不变量。
assert cache_header["config_hash"] == fingerprint(config)  # 用受控断言验证关键不变量。
assert fingerprint(MLAConfig(D_MODEL, N_HEADS, KV_RANK + 1, D_ROPE, D_VALUE)) != cache_header["config_hash"]  # 用受控断言验证关键不变量。


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
